In [ ]:
# The pre-requisite for this python file is ncat to be installed 
#docker exec -it <jupyter-lab-process-id>
#sudo apt-get install ncat

In [3]:
from pyspark.sql import SparkSession

spark=(SparkSession
.builder 
.appName("read from socket") 
.config("spark.streaming.stopGracefullyOnShutdown",True)
.master("local[*]") 
.getOrCreate()
      )
spark


In [4]:
#Read Input
stream_df=(spark.read.
           format("json").
           load("input/device_01.json")
          )

In [5]:
stream_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [8]:
#explode
from pyspark.sql.functions import explode

df_explode=stream_df.withColumn("data_devices",explode("data.devices"))
df_explode.printSchema()
# df_explode.show(truncate=False)

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [13]:
#flatten df
from pyspark.sql.functions import col
flatten_df=(df_explode
            .withColumn("device_id",col("data_devices.deviceid"))
            .withColumn("measure",col("data_devices.measure"))
            .withColumn("status",col("data_devices.status"))
            .withColumn("temperature",col("data_devices.temperature"))
            .drop("data")
            .drop("data_devices"))
# df_agg=df_explode.groupBy("word").agg(count(lit (1)).alias("CNT"))
flatten_df.printSchema()
flatten_df.show()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)

+----------+--------------------+-----------+--------------+--------------------+---------+-------+-------+-----------+
|customerId|             eventId|eventOffset|eventPublisher|           eventTime|device_id|measure| status|temperature|
+----------+--------------------+-----------+--------------+--------------------+---------+-------+-------+-----------+
|   CI00103|e3cb26d3-41b2-49a...|      10001|        device|2023-01-05 11:13:...|     D001|      C|  ERROR|         15|
|   CI00103|e3cb26d3-41b2-49a...|      10001|        device|2023-01-05 11:13:...|     D002|      C|SUCCESS|         16|
+----------+--------------

In [7]:
#WriteStream-update
df_agg.writeStream.format("console").outputMode("update").start().awaitTermination

<bound method StreamingQuery.awaitTermination of <pyspark.sql.streaming.StreamingQuery object at 0x779ae8ccbb50>>